# LeetCode 4095 — Count Frequency Staircase Subarrays II (Hard)

Bài toán LeetCode:

```python
class Solution:
    def countStaircase(self, nums: List[int]) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4095 (Weekly Contest 525 — câu 4, rating khoảng 2343).

Đây là bài đếm đoạn con với một điều kiện "kỳ lạ" về **tần suất theo thứ tự xuất hiện**. Điều kiện này **không** đơn điệu theo cách thông thường (kéo dài đoạn có thể từ "sai" thành "đúng"), nên kỹ thuật cửa sổ trượt quen thuộc không dùng được. Thay vào đó ta học:

- cách **kiểm tra tăng dần** (incremental check) trong `O(1)` mỗi bước;
- cách tìm các điều kiện **"đã hỏng thì không cứu được"** để **dừng sớm**;
- cách dùng **danh sách vị trí xuất hiện** của từng giá trị để biết trước đoạn hợp lệ phải kết thúc trong khoảng nào.

---

## Sau notebook này, bạn sẽ nắm được

1. Định nghĩa "bậc thang tần suất" và tính chất: độ dài đoạn luôn là **số tam giác** `1 + 2 + ... + k`.
2. Brute Force `O(n³)`.
3. Cách `O(n²)`: cố định điểm đầu, cập nhật tần suất, đếm số giá trị "đủ bậc", dừng khi có giá trị vượt bậc.
4. Cách nhanh: thêm điều kiện dừng "không đủ số lần xuất hiện" và tính trước **biên `[l, r)`** của điểm cuối.
5. Phân tích số bước (khoảng `n√n` trong trường hợp xấu), đo thực nghiệm.
6. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho mảng số nguyên `nums`. Với một đoạn con, liệt kê các **giá trị phân biệt theo thứ tự xuất hiện lần đầu** trong đoạn: `v1, v2, ..., vk`.

Đoạn được gọi là **bậc thang tần suất (frequency staircase)** nếu:

```text
v1 xuất hiện đúng 1 lần, v2 đúng 2 lần, ..., vk đúng k lần
```

Đếm số đoạn con bậc thang tần suất.

---

## Ví dụ 1

```python
nums = [5, 8, 8, 2, 2, 2]   ->   9
```

- 6 đoạn độ dài 1 (tần suất `[1]`).
- `[5, 8, 8]`: `5` một lần, `8` hai lần → `[1, 2]` ✔
- `[8, 2, 2]`: `[1, 2]` ✔
- Cả mảng: `5` một lần, `8` hai lần, `2` ba lần → `[1, 2, 3]` ✔

Tổng `6 + 3 = 9`.

## Ví dụ 2

```python
nums = [6, 6, 9, 9]   ->   5
```

4 đoạn độ dài 1, và `[6, 9, 9]` → `[1, 2]`. (`[6, 6]` có tần suất `[2]` — giá trị đầu tiên phải xuất hiện **1** lần → ✗.)

## Ví dụ 3

```python
nums = [2, 3, 4, 4, 3, 4]   ->   8
```

6 đoạn độ dài 1, `[3, 4, 4]` → `[1, 2]`, và cả mảng: `2` một lần, `3` hai lần, `4` ba lần → `[1, 2, 3]`. Để ý các lần xuất hiện **không cần liền nhau**.

---

## Ràng buộc

```text
1 <= nums.length <= 5 * 10^4
1 <= nums[i] <= 10^5
```

# 2. Kiến thức nền tảng

## 2.1. Kiểm tra một đoạn có phải bậc thang không

Duyệt đoạn, ghi nhận thứ tự xuất hiện lần đầu (`order`) và đếm tần suất (`cnt`). Kiểm tra `cnt[order[j]] == j + 1` với mọi `j`.

In [1]:
from typing import List


def is_staircase(seg: List[int]) -> bool:
    order = []
    cnt = {}
    for v in seg:
        if v not in cnt:
            order.append(v)
            cnt[v] = 0
        cnt[v] += 1
    return all(cnt[v] == j + 1 for j, v in enumerate(order))


print(is_staircase([5, 8, 8]))            # True
print(is_staircase([2, 3, 4, 4, 3, 4]))   # True
print(is_staircase([6, 6]))               # False
print(is_staircase([8, 8, 5]))            # False: 8 xuất hiện trước nên phải đúng 1 lần

True
True
False
False


## 2.2. Độ dài luôn là số tam giác

Đoạn bậc thang với `k` giá trị phân biệt có độ dài `1 + 2 + ... + k = k(k+1)/2` — một **số tam giác**: `1, 3, 6, 10, 15, ...`

Hệ quả quan trọng: với `n <= 5·10^4`, số giá trị phân biệt `k` không vượt quá khoảng `√(2n) ≈ 316`.

In [2]:
tri = [k * (k + 1) // 2 for k in range(1, 11)]
print("số tam giác:", tri)
k = 1
while (k + 1) * (k + 2) // 2 <= 5 * 10**4:
    k += 1
print("k lớn nhất khi n = 5·10^4:", k)

số tam giác: [1, 3, 6, 10, 15, 21, 28, 36, 45, 55]
k lớn nhất khi n = 5·10^4: 315


## 2.3. Điều kiện này **không** đơn điệu

Ở nhiều bài đếm đoạn con, "đoạn tốt thì đoạn con của nó cũng tốt" — nhờ đó dùng được cửa sổ trượt. Ở đây thì không:

- `[5, 8, 8]` tốt nhưng đoạn con `[5, 8]` (tần suất `[1, 1]`) không tốt;
- `[5, 8]` không tốt nhưng kéo dài thành `[5, 8, 8]` lại tốt.

Vì vậy ta phải nghĩ cách khác.

## 2.4. Danh sách vị trí xuất hiện

`positions[v]` = danh sách chỉ số mà giá trị `v` xuất hiện (tăng dần). Rất tiện để hỏi "lần xuất hiện thứ `t` của `v` (tính từ một vị trí) ở đâu?".

In [3]:
nums = [2, 3, 4, 4, 3, 4]
positions = {}
for i, v in enumerate(nums):
    positions.setdefault(v, []).append(i)
print(positions)

{2: [0], 3: [1, 4], 4: [2, 3, 5]}


# 3. Cách 1 — Brute Force: kiểm tra mọi đoạn

In [4]:
def count_bruteforce(nums: List[int]) -> int:
    n = len(nums)
    return sum(is_staircase(nums[i:j + 1]) for i in range(n) for j in range(i, n))


print(count_bruteforce([5, 8, 8, 2, 2, 2]))   # 9
print(count_bruteforce([6, 6, 9, 9]))         # 5
print(count_bruteforce([2, 3, 4, 4, 3, 4]))   # 8

9
5
8


`O(n²)` đoạn × `O(n)` kiểm tra = `O(n³)`.

---

# 4. Cách 2 — Cố định điểm đầu, kiểm tra tăng dần, dừng sớm

Cố định điểm đầu `left`, kéo `right` sang phải. Duy trì:

- `rank[v]`: thứ tự xuất hiện lần đầu của `v` trong đoạn (1, 2, 3, ...) — **không đổi** khi kéo dài;
- `cnt[v]`: tần suất hiện tại;
- `ok`: số giá trị đang có `cnt[v] == rank[v]` ("đủ bậc").

Đoạn là bậc thang ⇔ `ok == số giá trị phân biệt`.

Khi thêm `v = nums[right]`, chỉ `cnt[v]` thay đổi, nên cập nhật `ok` trong `O(1)`.

## Điều kiện dừng sớm 1: có giá trị **vượt bậc**

Nếu `cnt[v] > rank[v]`, kéo dài thêm chỉ làm `cnt[v]` tăng — không bao giờ quay về đúng `rank[v]`. Mọi đoạn dài hơn (cùng điểm đầu) đều hỏng → `break`.

In [5]:
def count_incremental(nums: List[int]) -> int:
    n = len(nums)
    total = 0
    for left in range(n):
        rank, cnt = {}, {}
        ok = 0
        for right in range(left, n):
            v = nums[right]
            if v not in rank:
                rank[v] = len(rank) + 1
                cnt[v] = 0
            cnt[v] += 1
            if cnt[v] > rank[v]:
                break                        # vượt bậc: không cứu được nữa
            if cnt[v] == rank[v]:
                ok += 1
            if ok == len(rank):
                total += 1
    return total


print(count_incremental([5, 8, 8, 2, 2, 2]))   # 9
print(count_incremental([6, 6, 9, 9]))         # 5
print(count_incremental([2, 3, 4, 4, 3, 4]))   # 8

9
5
8


## Mô phỏng với `left = 0`, `nums = [5, 8, 8, 2, 2, 2]`

| right | v | rank[v] | cnt[v] | ok / số giá trị | bậc thang? |
|---:|---:|---:|---:|---|:---:|
| 0 | 5 | 1 | 1 | 1 / 1 | ✔ |
| 1 | 8 | 2 | 1 | 1 / 2 | |
| 2 | 8 | 2 | 2 | 2 / 2 | ✔ |
| 3 | 2 | 3 | 1 | 2 / 3 | |
| 4 | 2 | 3 | 2 | 2 / 3 | |
| 5 | 2 | 3 | 3 | 3 / 3 | ✔ |

## Vẫn có trường hợp `O(n²)`

Nếu mọi phần tử **khác nhau**, không giá trị nào vượt bậc (mỗi giá trị chỉ có 1 lần), nên vòng trong chạy tới cuối mảng với **mọi** `left` → `n²/2` bước, dù chỉ có các đoạn độ dài 1 là hợp lệ.

In [6]:
def steps_incremental(nums: List[int]) -> int:
    n, steps = len(nums), 0
    for left in range(n):
        rank, cnt = {}, {}
        for right in range(left, n):
            steps += 1
            v = nums[right]
            if v not in rank:
                rank[v] = len(rank) + 1
                cnt[v] = 0
            cnt[v] += 1
            if cnt[v] > rank[v]:
                break
    return steps


for n_ in [500, 1000, 2000]:
    print(f"mảng {n_} phần tử khác nhau: {steps_incremental(list(range(n_))):,} bước")

mảng 500 phần tử khác nhau: 125,250 bước
mảng 1000 phần tử khác nhau: 500,500 bước


mảng 2000 phần tử khác nhau: 2,001,000 bước


# 5. Bước đột phá: điều kiện dừng thứ hai và biên `[l, r)`

## Điều kiện dừng sớm 2: giá trị thứ `d` **không đủ** `d` lần xuất hiện

Khi `right` gặp một giá trị **mới** (xuất hiện lần đầu trong đoạn), nó có thứ hạng `d`. Để đoạn có thể hợp lệ, giá trị này phải xuất hiện **đủ `d` lần** từ `left` trở đi. Nếu tổng số lần xuất hiện của nó trong `nums[left:]` ít hơn `d` thì **không đoạn nào** (cùng `left`, chứa vị trí này) hợp lệ — mọi đoạn dài hơn cũng chứa giá trị này với cùng thứ hạng `d` → `break`.

Với mảng toàn phần tử khác nhau: giá trị thứ 2 chỉ có 1 lần xuất hiện < 2 → dừng ngay sau 2 bước.

## Biên `[l, r)` cho điểm cuối

Khi giá trị mới `v` có thứ hạng `d`, gọi `p_t` là vị trí xuất hiện thứ `t` của `v` tính từ `left`. Để `v` xuất hiện **đúng** `d` lần trong `[left, right]`:

```text
right >= p_d        (đã gặp đủ d lần)
right <  p_{d+1}    (chưa gặp lần thứ d+1), nếu có
```

Ta duy trì:

- `l` = max của mọi `p_d` (điểm cuối nhỏ nhất để **mọi** giá trị đủ bậc),
- `r` = min của mọi `p_{d+1}` (điểm cuối phải **nhỏ hơn** — vượt qua là có giá trị vượt bậc).

Đoạn `[left, right]` là bậc thang ⇔ `l <= right < r`. Vòng lặp chạy khi `right < r`; mỗi bước chỉ cần kiểm tra `l <= right`.

## Lấy nhanh `p_t`

Duyệt `left` **từ phải sang trái**, mỗi lần thêm `left` vào cuối danh sách `lookup[nums[left]]`. Danh sách khi đó **giảm dần** và chỉ chứa các vị trí `>= left`; phần tử cuối là lần xuất hiện **đầu tiên** từ `left`:

```text
p_t = lookup[v][-t]
```

Và `right` là lần xuất hiện đầu tiên của `v` trong đoạn ⇔ `lookup[v][-1] == right`.

# 6. Cách 3 — Hai điều kiện dừng + biên `[l, r)`

In [7]:
def count_fast(nums: List[int]) -> int:
    n = len(nums)
    lookup = {}
    total = 0
    for left in range(n - 1, -1, -1):
        lookup.setdefault(nums[left], []).append(left)       # vị trí giảm dần, cuối là nhỏ nhất
        d, l, r = 0, -1, n
        right = left
        while right < r:
            idxs = lookup[nums[right]]
            if idxs[-1] == right:                             # giá trị mới, thứ hạng d + 1
                d += 1
                if len(idxs) < d:
                    break                                     # không đủ d lần xuất hiện
                l = max(l, idxs[-d])                          # phải tới lần thứ d
                if len(idxs) > d:
                    r = min(r, idxs[-d - 1])                  # không được tới lần thứ d + 1
            if l <= right:
                total += 1
            right += 1
    return total


print(count_fast([5, 8, 8, 2, 2, 2]))   # 9
print(count_fast([6, 6, 9, 9]))         # 5
print(count_fast([2, 3, 4, 4, 3, 4]))   # 8

9
5
8


## Mô phỏng với `left = 0`, `nums = [2, 3, 4, 4, 3, 4]`

Vị trí xuất hiện từ `left = 0`: `2 → [0]`, `3 → [1, 4]`, `4 → [2, 3, 5]`.

| right | v | mới? | d | cập nhật | l | r | `l <= right`? |
|---:|---:|:---:|---:|---|---:|---:|:---:|
| 0 | 2 | ✔ | 1 | `l = p_1(2) = 0`; không có lần 2 | 0 | 6 | ✔ đếm |
| 1 | 3 | ✔ | 2 | `l = max(0, p_2(3) = 4)` | 4 | 6 | |
| 2 | 4 | ✔ | 3 | `l = max(4, p_3(4) = 5)` | 5 | 6 | |
| 3 | 4 | | 3 | — | 5 | 6 | |
| 4 | 3 | | 3 | — | 5 | 6 | |
| 5 | 4 | | 3 | — | 5 | 6 | ✔ đếm |

Với `left = 0` đếm được 2 đoạn: `[2]` và cả mảng ✔.

In [8]:
def trace_left(nums: List[int], left: int):
    n = len(nums)
    lookup = {}
    for i in range(n - 1, left - 1, -1):
        lookup.setdefault(nums[i], []).append(i)
    d, l, r = 0, -1, n
    right = left
    print(f"left = {left}, vị trí từ left: { {v: idx[::-1] for v, idx in lookup.items()} }")
    while right < r:
        idxs = lookup[nums[right]]
        note = ""
        if idxs[-1] == right:
            d += 1
            if len(idxs) < d:
                print(f"right = {right}: giá trị mới {nums[right]} cần {d} lần nhưng chỉ có {len(idxs)} -> dừng")
                break
            l = max(l, idxs[-d])
            if len(idxs) > d:
                r = min(r, idxs[-d - 1])
            note = f"mới, d = {d}"
        counted = l <= right
        print(f"right = {right}, v = {nums[right]}: {note:10} l = {l}, r = {r} -> {'ĐẾM' if counted else '-'}")
        right += 1


trace_left([2, 3, 4, 4, 3, 4], 0)
print()
trace_left([2, 3, 4, 4, 3, 4], 1)

left = 0, vị trí từ left: {4: [2, 3, 5], 3: [1, 4], 2: [0]}
right = 0, v = 2: mới, d = 1 l = 0, r = 6 -> ĐẾM
right = 1, v = 3: mới, d = 2 l = 4, r = 6 -> -
right = 2, v = 4: mới, d = 3 l = 5, r = 6 -> -
right = 3, v = 4:            l = 5, r = 6 -> -
right = 4, v = 3:            l = 5, r = 6 -> -
right = 5, v = 4:            l = 5, r = 6 -> ĐẾM

left = 1, vị trí từ left: {4: [2, 3, 5], 3: [1, 4]}
right = 1, v = 3: mới, d = 1 l = 1, r = 4 -> ĐẾM
right = 2, v = 4: mới, d = 2 l = 3, r = 4 -> -
right = 3, v = 4:            l = 3, r = 4 -> ĐẾM


Với `left = 1`: giá trị `3` có thứ hạng 1 nên `r = p_2(3) = 4` — đoạn không được chạm tới số `3` thứ hai. Giá trị `4` thứ hạng 2 → `l = p_2(4) = 3`. Chỉ `right = 1` (đoạn `[3]`) và `right = 3` (đoạn `[3, 4, 4]`) được đếm, rồi vòng lặp dừng ở `right = 4 = r`.

# 7. Phiên bản đúng format LeetCode

In [9]:
from typing import List


class Solution:
    def countStaircase(self, nums: List[int]) -> int:
        n = len(nums)
        lookup = {}
        total = 0
        for left in range(n - 1, -1, -1):
            lookup.setdefault(nums[left], []).append(left)
            d, l, r = 0, -1, n
            right = left
            while right < r:
                idxs = lookup[nums[right]]
                if idxs[-1] == right:
                    d += 1
                    if len(idxs) < d:
                        break
                    if idxs[-d] > l:
                        l = idxs[-d]
                    if len(idxs) > d and idxs[-d - 1] < r:
                        r = idxs[-d - 1]
                if l <= right:
                    total += 1
                right += 1
        return total

In [10]:
sol = Solution()
tests = [
    ([5, 8, 8, 2, 2, 2], 9, "ví dụ 1"),
    ([6, 6, 9, 9], 5, "ví dụ 2"),
    ([2, 3, 4, 4, 3, 4], 8, "ví dụ 3"),
    ([7], 1, "một phần tử"),
    ([1, 1, 1, 1], 4, "toàn bằng nhau"),
    ([1, 2, 3, 4], 4, "toàn khác nhau"),
    ([1, 2, 2, 3, 3, 3, 4, 4, 4, 4], 15, "bậc thang 4 bậc"),
]
for nums, expected, name in tests:
    result = sol.countStaircase(nums)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:20} -> {result}")

PASS | ví dụ 1              -> 9
PASS | ví dụ 2              -> 5
PASS | ví dụ 3              -> 8
PASS | một phần tử          -> 1
PASS | toàn bằng nhau       -> 4
PASS | toàn khác nhau       -> 4
PASS | bậc thang 4 bậc      -> 15


# 8. Vì sao đúng?

Cố định `left`. Gọi `v1, v2, ...` là các giá trị theo thứ tự xuất hiện lần đầu từ `left`.

1. Đoạn `[left, right]` chứa đúng `v1..vd` (với `d` = số giá trị mới đã gặp tới `right`). Nó là bậc thang ⇔ với mọi `j <= d`: `vj` xuất hiện đúng `j` lần ⇔ `p_j(vj) <= right < p_{j+1}(vj)` ⇔ `l <= right < r` với `l = max p_j(vj)`, `r = min p_{j+1}(vj)`.
2. Khi `right >= r`: một giá trị đã vượt bậc, mọi đoạn dài hơn đều hỏng → dừng đúng.
3. Khi giá trị mới `vd` có ít hơn `d` lần xuất hiện trong `nums[left:]`: mọi đoạn chứa vị trí này có `vd` với thứ hạng `d` nhưng tần suất `< d` → hỏng; đoạn dài hơn cũng vậy → dừng đúng.
4. Mỗi đoạn hợp lệ được đếm đúng một lần (theo cặp `(left, right)`).

# 9. Độ phức tạp

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---:|---:|
| 1. Brute Force | kiểm tra mọi đoạn | `O(n³)` | `O(n)` |
| 2. Tăng dần + dừng khi vượt bậc | `ok` đếm giá trị đủ bậc | `O(n²)` (mảng toàn khác nhau) | `O(n)` |
| 3. Hai điều kiện dừng + `[l, r)` | danh sách vị trí, biên điểm cuối | khoảng `O(n√n)` trường hợp xấu | `O(n)` |

**Về số bước của cách 3.** Mỗi lần gặp giá trị mới thứ `d`, giá trị đó phải có ít nhất `d` lần xuất hiện, nên số giá trị phân biệt trong một lượt duyệt không quá khoảng `√(2n)`. Lời giải tham khảo của bài đánh giá độ phức tạp là `O(n√n)`. Dưới đây ta **đo số bước thực tế** trên vài kiểu dữ liệu, trong đó có một mảng được dựng để "khó" cho thuật toán: `[1..d] + [2..d] + [3..d] + ...` (giá trị `j` xuất hiện `j` lần, rải đều).

In [11]:
import random
import time


def steps_fast(nums: List[int]) -> int:
    n = len(nums)
    lookup = {}
    steps = 0
    for left in range(n - 1, -1, -1):
        lookup.setdefault(nums[left], []).append(left)
        d, l, r = 0, -1, n
        right = left
        while right < r:
            steps += 1
            idxs = lookup[nums[right]]
            if idxs[-1] == right:
                d += 1
                if len(idxs) < d:
                    break
                l = max(l, idxs[-d])
                if len(idxs) > d:
                    r = min(r, idxs[-d - 1])
            right += 1
    return steps


random.seed(4095)
N = 5 * 10**4
d = 315
cases = {
    "ngẫu nhiên, 3 giá trị": [random.randint(1, 3) for _ in range(N)],
    "ngẫu nhiên, 300 giá trị": [random.randint(1, 300) for _ in range(N)],
    "toàn khác nhau": list(range(N)),
    "toàn bằng nhau": [1] * N,
    "rải tam giác (khó)": [v for t in range(1, d + 1) for v in range(t, d + 1)],
}
for name, arr in cases.items():
    t0 = time.perf_counter()
    s = steps_fast(arr)
    print(f"{name:26} n = {len(arr):>6}: {s:>10,} bước ({time.perf_counter() - t0:.2f}s), "
          f"n·√n ≈ {int(len(arr) * len(arr) ** 0.5):,}")

ngẫu nhiên, 3 giá trị      n =  50000:    123,292 bước (0.03s), n·√n ≈ 11,180,339


ngẫu nhiên, 300 giá trị    n =  50000:  3,672,038 bước (0.75s), n·√n ≈ 11,180,339
toàn khác nhau             n =  50000:     99,999 bước (0.02s), n·√n ≈ 11,180,339
toàn bằng nhau             n =  50000:     50,000 bước (0.01s), n·√n ≈ 11,180,339


rải tam giác (khó)         n =  49770: 10,517,431 bước (1.50s), n·√n ≈ 11,103,284


Số bước đều không vượt quá khoảng `n·√n`; mảng "rải tam giác" được dựng để chạm gần mức này — khớp với đánh giá `O(n√n)`. Ở đó mỗi điểm đầu nằm trong khối đầu tiên phải đi khoảng `√n` bước mới gặp lại giá trị của chính nó.

# 10. Đối chiếu ba cách trên dữ liệu ngẫu nhiên

In [12]:
random.seed(2026)
for _ in range(3000):
    nums = [random.randint(1, 4) for _ in range(random.randint(1, 12))]
    expected = count_bruteforce(nums)
    assert count_incremental(nums) == expected, nums
    assert count_fast(nums) == expected, nums
    assert sol.countStaircase(nums) == expected, nums
print("3000 bộ test ngẫu nhiên: 4 hàm cho cùng kết quả!")

3000 bộ test ngẫu nhiên: 4 hàm cho cùng kết quả!


# 11. Những lỗi dễ mắc

## Lỗi 1 — Dùng cửa sổ trượt hai con trỏ

Vì điều kiện **không đơn điệu** (mục 2.3), kiểu "mở rộng phải, thu hẹp trái khi hỏng" sẽ bỏ sót đoạn. Ví dụ `[5, 8, 8]`: tại `right = 1`, đoạn `[5, 8]` hỏng; thu hẹp trái thành `[8]` rồi tiếp tục sẽ không bao giờ đếm được `[5, 8, 8]`.

## Lỗi 2 — Cập nhật biên với cả những lần xuất hiện **lặp lại**

Biên `l, r` chỉ được cập nhật khi gặp giá trị **mới** (lần đầu trong đoạn), dùng thứ hạng của giá trị đó. Nếu cập nhật ở mọi bước với `d` hiện tại, các giá trị cũ bị áp sai thứ hạng.

In [13]:
def wrong_update_every_step(nums):
    n = len(nums)
    lookup = {}
    total = 0
    for left in range(n - 1, -1, -1):
        lookup.setdefault(nums[left], []).append(left)
        d, l, r = 0, -1, n
        right = left
        while right < r:
            idxs = lookup[nums[right]]
            if idxs[-1] == right:
                d += 1
            if len(idxs) < d:                    # SAI: các dòng dưới chạy cả khi v không mới
                break
            if len(idxs) > d:
                r = min(r, idxs[-d - 1])
            l = max(l, idxs[-d])
            if l <= right:
                total += 1
            right += 1
    return total


print("Sai :", wrong_update_every_step([2, 3, 4, 4, 3, 4]))
print("Đúng:", sol.countStaircase([2, 3, 4, 4, 3, 4]))

Sai : 7
Đúng: 8


## Lỗi 3 — Quên rằng thứ tự là theo **lần xuất hiện đầu tiên trong đoạn**

`[8, 8, 5]`: giá trị đầu tiên là `8` (phải xuất hiện 1 lần) → ✗, dù tập tần suất `{1, 2}` "trông giống" bậc thang. Đừng sắp xếp tần suất rồi so với `1..k`.

In [14]:
def wrong_sorted_freq(seg):
    from collections import Counter
    return sorted(Counter(seg).values()) == list(range(1, len(set(seg)) + 1))   # SAI


print("Sai :", wrong_sorted_freq([8, 8, 5]))
print("Đúng:", is_staircase([8, 8, 5]))

Sai : True
Đúng: False


## Lỗi 4 — Thiếu điều kiện dừng "không đủ lần xuất hiện"

Code vẫn đúng nhưng chạy `O(n²)` với mảng toàn khác nhau (mục 4) → TLE với `n = 5·10^4`.

## Lỗi 5 — Duyệt `left` từ trái sang phải mà vẫn dùng `lookup[v][-t]`

Mẹo "danh sách giảm dần, phần tử cuối là lần xuất hiện đầu tiên từ `left`" chỉ đúng khi duyệt `left` **từ phải sang trái** và **thêm dần** vị trí. Duyệt ngược chiều thì danh sách chứa cả vị trí `< left`.

# 12. Mẹo nhận diện

1. Đếm đoạn con với điều kiện về **tần suất** → thử cố định một đầu, cập nhật tần suất tăng dần, giữ một biến đếm "số giá trị thỏa" để kiểm tra trong `O(1)`.
2. Điều kiện không đơn điệu → tìm các **điều kiện "đã hỏng thì không cứu được"** để dừng sớm.
3. Cần "lần xuất hiện thứ `t` của `v` sau vị trí `x`" → **danh sách vị trí**; duyệt `x` ngược chiều để danh sách luôn gọn.
4. Tổng tần suất `1 + 2 + ... + k` → `k = O(√n)` — gợi ý độ phức tạp có căn.

Các bài liên quan: *Count Frequency Staircase Subarrays I* (4094 — phiên bản `n` nhỏ), *Count Complete Substrings* (2953), *Subarrays with K Different Integers* (992).

# 13. Biến thể

## Biến thể 1 — Đoạn bậc thang dài nhất

Dùng cách 3, mỗi lần đếm thì cập nhật `right - left + 1`.

In [15]:
def longest_staircase(nums: List[int]) -> int:
    n = len(nums)
    lookup = {}
    best = 0
    for left in range(n - 1, -1, -1):
        lookup.setdefault(nums[left], []).append(left)
        d, l, r = 0, -1, n
        right = left
        while right < r:
            idxs = lookup[nums[right]]
            if idxs[-1] == right:
                d += 1
                if len(idxs) < d:
                    break
                l = max(l, idxs[-d])
                if len(idxs) > d:
                    r = min(r, idxs[-d - 1])
            if l <= right:
                best = max(best, right - left + 1)
            right += 1
    return best


print(longest_staircase([5, 8, 8, 2, 2, 2]))   # 6
print(longest_staircase([6, 6, 9, 9]))         # 3

6
3


## Biến thể 2 — Chỉ đếm đoạn có đúng `k` bậc

Thêm điều kiện `d == k` khi đếm (và có thể dừng khi `d > k`).

In [16]:
def count_exact_k(nums: List[int], k: int) -> int:
    n = len(nums)
    lookup = {}
    total = 0
    for left in range(n - 1, -1, -1):
        lookup.setdefault(nums[left], []).append(left)
        d, l, r = 0, -1, n
        right = left
        while right < r:
            idxs = lookup[nums[right]]
            if idxs[-1] == right:
                d += 1
                if d > k or len(idxs) < d:
                    break
                l = max(l, idxs[-d])
                if len(idxs) > d:
                    r = min(r, idxs[-d - 1])
            if l <= right and d == k:
                total += 1
            right += 1
    return total


print([count_exact_k([5, 8, 8, 2, 2, 2], k) for k in (1, 2, 3)])   # [6, 2, 1]

[6, 2, 1]


# 14. Template tư duy

```python
lookup = {}
for left in range(n - 1, -1, -1):              # duyệt ngược để lookup[v][-t] là lần thứ t từ left
    lookup.setdefault(nums[left], []).append(left)
    d, l, r = 0, -1, n
    right = left
    while right < r:                            # r: vượt qua là có giá trị vượt bậc
        idxs = lookup[nums[right]]
        if idxs[-1] == right:                   # giá trị mới, thứ hạng d
            d += 1
            if len(idxs) < d: break             # không đủ số lần
            l = max(l, idxs[-d])
            if len(idxs) > d: r = min(r, idxs[-d - 1])
        if l <= right: count += 1
        right += 1
```

# 15. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Liệt kê mọi đoạn bậc thang của `nums = [1, 2, 2, 3, 1, 3]` (gợi ý: có 7 đoạn). Kiểm tra bằng cell dưới.

In [17]:
# Kiểm tra Bài 1
arr = [1, 2, 2, 3, 1, 3]
found = [(i, j, arr[i:j + 1]) for i in range(len(arr)) for j in range(i, len(arr)) if is_staircase(arr[i:j + 1])]
for item in found:
    print(item)
print("Tổng:", len(found))

(0, 0, [1])
(0, 2, [1, 2, 2])
(1, 1, [2])
(2, 2, [2])
(3, 3, [3])
(4, 4, [1])
(5, 5, [3])
Tổng: 7


## Bài 2 — Dự đoán output

```python
print(is_staircase([4, 9, 9, 4]))
print(is_staircase([4, 9, 4, 9]))
print(is_staircase([3, 1, 1, 2, 2, 2]))
```

In [18]:
print(is_staircase([4, 9, 9, 4]))
print(is_staircase([4, 9, 4, 9]))
print(is_staircase([3, 1, 1, 2, 2, 2]))

False
False
True


## Bài 3 — Hoàn thiện cách 2 (tăng dần + dừng khi vượt bậc)

In [19]:
def exercise_3_incremental(nums: List[int]) -> int:
    n = len(nums)
    total = 0
    for left in range(n):
        rank, cnt, ok = {}, {}, 0
        for right in range(left, n):
            # TODO 1: gán rank cho giá trị mới, tăng cnt
            # TODO 2: nếu cnt > rank thì break; nếu cnt == rank thì ok += 1
            # TODO 3: nếu ok == số giá trị phân biệt thì total += 1
            pass
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_incremental([5, 8, 8, 2, 2, 2]) == 9

## Bài 4 — Liệt kê các độ dài có thể

Viết hàm trả về tập các **độ dài** của mọi đoạn bậc thang trong mảng. Kiểm tra rằng mọi độ dài đều là số tam giác.

```python
staircase_lengths([5, 8, 8, 2, 2, 2])   -> {1, 3, 6}
```

In [20]:
def exercise_4_staircase_lengths(nums: List[int]) -> set:
    # TODO
    raise NotImplementedError


# assert exercise_4_staircase_lengths([5, 8, 8, 2, 2, 2]) == {1, 3, 6}

## Bài 5 — Đoạn "cân bằng tần suất"

Đếm số đoạn mà **mọi** giá trị phân biệt có **cùng** tần suất (`n` nhỏ, `O(n²)` là được).

```python
count_equal_freq([1, 2, 1, 2])   -> 8     # 4 đoạn độ dài 1, [1,2], [2,1], [1,2], [1,2,1,2]
count_equal_freq([1, 1, 2])      -> 5     # [1],[1],[2],[1,1],[1,2]
```

Gợi ý: duy trì `cnt[v]` và `freq_of_freq[f]` = số giá trị có tần suất `f`.

In [21]:
def exercise_5_count_equal_freq(nums: List[int]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_count_equal_freq([1, 2, 1, 2]) == 8
# assert exercise_5_count_equal_freq([1, 1, 2]) == 5

## Bài 6 — Mini challenge: bậc thang "ngược"

Đếm số đoạn mà giá trị thứ `j` (theo thứ tự xuất hiện đầu tiên) xuất hiện đúng `k - j + 1` lần (tức tần suất `k, k-1, ..., 1`). Viết lời giải `O(n²)` có dừng sớm hợp lý và kiểm tra với brute force.

```python
count_reverse_staircase([2, 2, 5])        -> 4     # [2],[2],[5],[2,2,5]
count_reverse_staircase([7, 7, 7, 1, 1, 3]) -> 9
```

In [22]:
def exercise_6_count_reverse_staircase(nums: List[int]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_6_count_reverse_staircase([2, 2, 5]) == 4

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: `if v not in rank: rank[v] = len(rank) + 1; cnt[v] = 0`.
- Mức 2: `cnt[v] += 1`; `if cnt[v] > rank[v]: break`.
- Mức 3: `if cnt[v] == rank[v]: ok += 1`; `if ok == len(rank): total += 1`.

**Bài 4**

- Mức 1: dùng cách 3, khi đếm thì thêm `right - left + 1` vào một `set`.
- Mức 2: số tam giác: `8·L + 1` là số chính phương.
- Mức 3: `all(int((8*L+1) ** 0.5) ** 2 == 8*L + 1 for L in lengths)`.

**Bài 5**

- Mức 1: khi `cnt[v]` đổi từ `f` sang `f+1`: giảm `freq_of_freq[f]`, tăng `freq_of_freq[f+1]`.
- Mức 2: mọi giá trị cùng tần suất `f` ⇔ `freq_of_freq[f] == số giá trị phân biệt`.
- Mức 3: `f = cnt[v]` của giá trị vừa thêm là ứng viên duy nhất cần kiểm tra.

**Bài 6**

- Mức 1: số bậc `k` chưa biết trước, nhưng giá trị đầu tiên có tần suất `k` — tức là `k` = tần suất của `v1` ở cuối đoạn.
- Mức 2: kiểm tra từng đoạn bằng hàm `is_reverse` với `O(1)` cập nhật là phức tạp; cách đơn giản: với mỗi `left`, duy trì `order` và `cnt`, kiểm tra trực tiếp khi độ dài là số tam giác.
- Mức 3: dừng khi độ dài vượt `k(k+1)/2` với `k` = số giá trị phân biệt tối đa có thể (√(2n)).

# 16. Đáp án tham khảo

## Đáp án Bài 1

7 đoạn: 6 đoạn độ dài 1 và `[1, 2, 2]` (chỉ số 0..2). Các đoạn như `[2, 3, 1, 3]`? → `2` một lần, `3` hai lần, nhưng `1` xuất hiện lần đầu ở vị trí thứ 3 với tần suất 1 ≠ 3 → ✗. Hãy đối chiếu với danh sách in ra ở cell trên.

## Đáp án Bài 2

```text
False    (4 xuất hiện 2 lần, nhưng là giá trị đầu tiên)
False
True     (3 một lần, 1 hai lần, 2 ba lần)
```

In [23]:
# Đáp án Bài 3
def answer_3_incremental(nums: List[int]) -> int:
    n = len(nums)
    total = 0
    for left in range(n):
        rank, cnt, ok = {}, {}, 0
        for right in range(left, n):
            v = nums[right]
            if v not in rank:
                rank[v] = len(rank) + 1
                cnt[v] = 0
            cnt[v] += 1
            if cnt[v] > rank[v]:
                break
            if cnt[v] == rank[v]:
                ok += 1
            if ok == len(rank):
                total += 1
    return total


assert answer_3_incremental([5, 8, 8, 2, 2, 2]) == 9
assert answer_3_incremental([1, 2, 2, 3, 1, 3]) == 7
print("Bài 3: PASS")

Bài 3: PASS


In [24]:
# Đáp án Bài 4
def answer_4_staircase_lengths(nums: List[int]) -> set:
    n = len(nums)
    lookup = {}
    lengths = set()
    for left in range(n - 1, -1, -1):
        lookup.setdefault(nums[left], []).append(left)
        d, l, r = 0, -1, n
        right = left
        while right < r:
            idxs = lookup[nums[right]]
            if idxs[-1] == right:
                d += 1
                if len(idxs) < d:
                    break
                l = max(l, idxs[-d])
                if len(idxs) > d:
                    r = min(r, idxs[-d - 1])
            if l <= right:
                lengths.add(right - left + 1)
            right += 1
    return lengths


assert answer_4_staircase_lengths([5, 8, 8, 2, 2, 2]) == {1, 3, 6}
random.seed(4)
for _ in range(300):
    arr = [random.randint(1, 3) for _ in range(random.randint(1, 15))]
    for L in answer_4_staircase_lengths(arr):
        assert int((8 * L + 1) ** 0.5) ** 2 == 8 * L + 1
print("Bài 4: PASS (mọi độ dài đều là số tam giác)")

Bài 4: PASS (mọi độ dài đều là số tam giác)


In [25]:
# Đáp án Bài 5
def answer_5_count_equal_freq(nums: List[int]) -> int:
    n = len(nums)
    total = 0
    for left in range(n):
        cnt, ff = {}, {}
        for right in range(left, n):
            v = nums[right]
            f = cnt.get(v, 0)
            if f:
                ff[f] -= 1
            cnt[v] = f + 1
            ff[f + 1] = ff.get(f + 1, 0) + 1
            if ff[f + 1] == len(cnt):
                total += 1
    return total


def brute_equal_freq(nums):
    from collections import Counter
    n = len(nums)
    return sum(len(set(Counter(nums[i:j + 1]).values())) == 1 for i in range(n) for j in range(i, n))


assert answer_5_count_equal_freq([1, 2, 1, 2]) == 8
assert answer_5_count_equal_freq([1, 1, 2]) == 5
random.seed(5)
for _ in range(500):
    arr = [random.randint(1, 3) for _ in range(random.randint(1, 10))]
    assert answer_5_count_equal_freq(arr) == brute_equal_freq(arr)
print("Bài 5: PASS (kể cả 500 test ngẫu nhiên)")

Bài 5: PASS (kể cả 500 test ngẫu nhiên)


In [26]:
# Đáp án Bài 6
def is_reverse_staircase(seg):
    order, cnt = [], {}
    for v in seg:
        if v not in cnt:
            order.append(v)
            cnt[v] = 0
        cnt[v] += 1
    k = len(order)
    return all(cnt[v] == k - j for j, v in enumerate(order))


def answer_6_count_reverse_staircase(nums: List[int]) -> int:
    n = len(nums)
    kmax = 1
    while (kmax + 1) * (kmax + 2) // 2 <= n:
        kmax += 1
    limit = kmax * (kmax + 1) // 2                 # đoạn hợp lệ không dài hơn số tam giác lớn nhất
    total = 0
    for left in range(n):
        order, cnt = [], {}
        for right in range(left, min(n, left + limit)):
            v = nums[right]
            if v not in cnt:
                order.append(v)
                cnt[v] = 0
            cnt[v] += 1
            k = len(order)
            if right - left + 1 == k * (k + 1) // 2 and all(cnt[w] == k - j for j, w in enumerate(order)):
                total += 1
    return total


def brute_reverse(nums):
    n = len(nums)
    return sum(is_reverse_staircase(nums[i:j + 1]) for i in range(n) for j in range(i, n))


assert answer_6_count_reverse_staircase([2, 2, 5]) == 4
assert answer_6_count_reverse_staircase([7, 7, 7, 1, 1, 3]) == 9
random.seed(6)
for _ in range(500):
    arr = [random.randint(1, 3) for _ in range(random.randint(1, 12))]
    assert answer_6_count_reverse_staircase(arr) == brute_reverse(arr), arr
print("Bài 6: PASS (khớp brute force trên 500 test)")

Bài 6: PASS (khớp brute force trên 500 test)


# 17. Tổng kết

**Count Frequency Staircase Subarrays II** dạy cách xử lý điều kiện **không đơn điệu**:

1. Cố định điểm đầu, cập nhật tăng dần.
2. Hai điều kiện dừng: **vượt bậc** (không cứu được) và **không đủ số lần xuất hiện** (không bao giờ đạt).
3. Biên điểm cuối `[l, r)` tính từ **danh sách vị trí**: `l = max p_d`, `r = min p_{d+1}`.
4. Số giá trị phân biệt của một bậc thang chỉ `O(√n)` → số bước khoảng `n√n`.

## Độ phức tạp

```text
Time:  khoảng O(n · √n)
Space: O(n)
```

# 18. Checklist trước khi submit

- [ ] Thứ tự giá trị là theo **lần xuất hiện đầu tiên trong đoạn**, không sắp xếp tần suất?
- [ ] Chỉ cập nhật `l, r` khi gặp giá trị **mới**?
- [ ] Có điều kiện dừng "không đủ `d` lần xuất hiện" (tránh `O(n²)` với mảng toàn khác nhau)?
- [ ] Duyệt `left` từ **phải sang trái** khi dùng `lookup[v][-t]`?
- [ ] Đếm khi `l <= right` (vòng lặp đã đảm bảo `right < r`)?
- [ ] Đã test: một phần tử, toàn bằng nhau, toàn khác nhau, bậc thang hoàn chỉnh?